# CropShield: logistic heat-related crop-loss risk model

This notebook tests whether a crop-specific logistic regression can improve on the **days at 35°C or hotter** baseline. It predicts the probability of a county-crop season ending at least 10% below its historical yield trend, using only heat, drought, and soil information available by the advisory date.

- Corn advisory date: **July 15**
- Soybean advisory date: **August 1**
- Model-selection period: **2000–2018**
- Locked unseen evaluation period: **2019–2025**
- Current scoring period: **2026** (no accuracy claim until yields are available)

> Important: the modeled score is a probability of a below-trend yield outcome under observed heat, drought, and soil conditions. It is not a causal estimate of heat alone and it is not an exact yield-loss forecast.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss, confusion_matrix, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Run this notebook from the repository root or its notebooks/ folder.')

sys.path.insert(0, str(ROOT))
from src.data_loader import load_daily_weather, load_master, load_soil, load_weekly_drought

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')


## 1. Lock the study design

`LOSS_ANOMALY_THRESHOLD = -10` defines a meaningful loss as a final yield at least 10% below the supplied county-crop yield trend. The supplied `yield_anomaly_pct` is used as the target for this hackathon model. The original master build estimates that trend using the full available history, so this target construction is documented as a limitation; the feature engineering and evaluation split below remain time-safe.

In [ ]:
STATE = 'IA'
CROPS = ('CORN', 'SOYBEANS')

# These dates define what would have been knowable at the time of an advisory.
SEASON_START = (5, 1)
ADVISORY_CUTOFFS = {
    'CORN': (7, 15),
    'SOYBEANS': (8, 1),
}

TRAIN_YEARS = list(range(2000, 2019))
TEST_YEARS = list(range(2019, 2026))
CURRENT_YEARS = [2026]
LOSS_ANOMALY_THRESHOLD = -10.0

# Small, interpretable candidate grid. C controls L2 regularization strength.
C_GRID = [0.01, 0.1, 1.0, 10.0]
MODEL_PROBABILITY_CANDIDATES = np.round(np.arange(0.05, 0.96, 0.01), 2)
FAR_TOLERANCE = 0.005
MODEL_VERSION = 'logistic_heat_loss_v1'

# Keep False while developing. The final export cell only writes files when set to True.
WRITE_OUTPUTS = False


## 2. Load sources and build a static soil context

`aws_100cm_mm` is soil water-storage capacity, not a learned soil-risk score. `low_soil_storage` is an Iowa-tercile indicator that lets the model test whether heat historically matters more in lower-storage counties.

In [ ]:
weather = load_daily_weather().copy()
drought = load_weekly_drought().copy()
soil = load_soil().copy()
master = load_master().copy()

for frame in (weather, drought, soil, master):
    frame['fips'] = frame['fips'].astype(str).str.zfill(5)

weather['date'] = pd.to_datetime(weather['date'])
drought['map_date'] = pd.to_datetime(drought['map_date'])

ia_weather = weather.loc[weather['state_alpha'].eq(STATE)].copy()
ia_drought = drought.loc[drought['state_alpha'].eq(STATE)].copy()
ia_soil = soil.loc[soil['state_alpha'].eq(STATE)].copy()
ia_master = master.loc[
    master['state_alpha'].eq(STATE) & master['crop'].isin(CROPS)
].copy()

soil_context = ia_soil[['fips', 'aws_100cm_mm']].drop_duplicates('fips').dropna().copy()
soil_context['soil_storage_group'] = pd.qcut(
    soil_context['aws_100cm_mm'].rank(method='first'),
    q=3,
    labels=['LOW', 'MEDIUM', 'HIGH'],
).astype(str)
soil_context['low_soil_storage'] = soil_context['soil_storage_group'].eq('LOW').astype(int)

county_lookup = (
    ia_drought.sort_values('map_date')
    .drop_duplicates('fips', keep='last')
    [['fips', 'county_name', 'state_alpha']]
)

print(f'Iowa daily-weather rows: {len(ia_weather):,}')
print(f'Iowa weekly-drought rows: {len(ia_drought):,}')
print(f'Iowa soil counties: {soil_context.fips.nunique()}')
print(f'Iowa master rows for modeled crops: {len(ia_master):,}')
display(soil_context.groupby('soil_storage_group', observed=True)['aws_100cm_mm'].agg(['count', 'min', 'median', 'max']))


## 3. Create time-safe advisory features

Every weather and drought aggregation stops on the crop-specific advisory date. This prevents full-season conditions or post-advisory information from leaking into model inputs.

In [ ]:
# Project drought scale: 0=None, 1=D0, 2=D1, 3=D2, 4=D3, 5=D4.
drought_scored = ia_drought.copy()
drought_scored['weekly_severity'] = 0
for severity, column in enumerate(['D0', 'D1', 'D2', 'D3', 'D4'], start=1):
    drought_scored.loc[drought_scored[column].gt(1), 'weekly_severity'] = severity
drought_scored['d2plus_week'] = drought_scored[['D2', 'D3', 'D4']].sum(axis=1).gt(1).astype(int)

def build_observed_feature_panel(years):
    """Build one county × crop × year row using only data through its advisory cutoff."""
    parts = []
    for crop, (month, day) in ADVISORY_CUTOFFS.items():
        for year in years:
            season_start = pd.Timestamp(year=year, month=SEASON_START[0], day=SEASON_START[1])
            as_of_date = pd.Timestamp(year=year, month=month, day=day)

            weather_window = ia_weather.loc[ia_weather['date'].between(season_start, as_of_date)]
            weather_features = (
                weather_window.assign(hot35_day=weather_window['tmax_c'].ge(35).astype(int))
                .groupby('fips', as_index=False)
                .agg(
                    hot35_days_to_date=('hot35_day', 'sum'),
                    weather_through_date=('date', 'max'),
                )
            )

            drought_window = drought_scored.loc[drought_scored['map_date'].between(season_start, as_of_date)]
            drought_features = (
                drought_window.groupby('fips', as_index=False)
                .agg(
                    d2plus_weeks_to_date=('d2plus_week', 'sum'),
                    max_drought_severity_to_date=('weekly_severity', 'max'),
                    drought_through_date=('map_date', 'max'),
                )
            )

            features = weather_features.merge(drought_features, on='fips', how='outer', validate='one_to_one')
            features['crop'] = crop
            features['year'] = year
            features['as_of_date'] = as_of_date
            parts.append(features)

    features = pd.concat(parts, ignore_index=True)
    features = (
        features
        .merge(county_lookup, on='fips', how='left', validate='many_to_one')
        .merge(soil_context, on='fips', how='left', validate='many_to_one')
    )
    features['hot35_x_drought'] = features['hot35_days_to_date'] * features['d2plus_weeks_to_date']
    features['hot35_x_low_storage'] = features['hot35_days_to_date'] * features['low_soil_storage']
    return features

all_features = build_observed_feature_panel(TRAIN_YEARS + TEST_YEARS + CURRENT_YEARS)
assert not all_features.duplicated(['fips', 'crop', 'year']).any()
display(all_features.head())
display(all_features.groupby(['crop', 'year']).size().unstack(0).tail())


## 4. Attach historical yield outcomes

The model never receives final yield anomaly as an input. It only receives it as the historical label it is trying to learn from. Rows with unreported yields cannot be used for fitting or evaluation.

In [ ]:
outcomes = ia_master.loc[
    ia_master['year'].isin(TRAIN_YEARS + TEST_YEARS)
    & ia_master['yield_status'].eq('reported')
    & ia_master['yield_anomaly_pct'].notna(),
    ['fips', 'crop', 'year', 'yield_per_acre', 'yield_anomaly_pct'],
].copy()
outcomes['loss_flag'] = outcomes['yield_anomaly_pct'].le(LOSS_ANOMALY_THRESHOLD).astype(int)

model_panel = all_features.merge(
    outcomes,
    on=['fips', 'crop', 'year'],
    how='inner',
    validate='one_to_one',
)

FEATURE_COLUMNS = [
    'hot35_days_to_date',
    'd2plus_weeks_to_date',
    'max_drought_severity_to_date',
    'aws_100cm_mm',
    'hot35_x_drought',
    'hot35_x_low_storage',
]

assert set(FEATURE_COLUMNS).issubset(model_panel.columns)
assert model_panel['loss_flag'].isin([0, 1]).all()

coverage = (
    model_panel.groupby(['crop', 'year'], as_index=False)
    .agg(rows=('loss_flag', 'size'), losses=('loss_flag', 'sum'), loss_rate=('loss_flag', 'mean'))
)
display(coverage.tail(16))
display(model_panel.groupby('crop')['loss_flag'].agg(rows='size', losses='sum', loss_rate='mean'))


## 5. Modeling and evaluation helpers

The logistic pipeline imputes missing feature values using the training fold only, standardizes numeric values, and applies L2 regularization. Validation folds respect time: a fold can only train on years before its validation year.

In [ ]:
def make_pipeline(C):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=C, penalty='l2', solver='lbfgs', max_iter=2000)),
    ])

def expanding_year_splits(frame, years):
    """Yield expanding train / one-year validation splits with both target classes in training."""
    valid_years = sorted(set(years))
    candidate_validation_years = valid_years[7::2] + [valid_years[-1]]
    splits = []
    for validation_year in sorted(set(candidate_validation_years)):
        train_index = frame.index[frame['year'].lt(validation_year)].to_numpy()
        validation_index = frame.index[frame['year'].eq(validation_year)].to_numpy()
        train_target = frame.loc[train_index, 'loss_flag']
        if len(validation_index) and train_target.nunique() == 2:
            splits.append((train_index, validation_index))
    if not splits:
        raise ValueError('No valid expanding-year splits were created. Check target coverage.')
    return splits

def out_of_fold_probabilities(frame, C):
    predictions = []
    for train_index, validation_index in expanding_year_splits(frame, TRAIN_YEARS):
        pipeline = make_pipeline(C)
        pipeline.fit(frame.loc[train_index, FEATURE_COLUMNS], frame.loc[train_index, 'loss_flag'])
        fold = frame.loc[validation_index, ['year', 'loss_flag', 'hot35_days_to_date']].copy()
        fold['model_probability'] = pipeline.predict_proba(frame.loc[validation_index, FEATURE_COLUMNS])[:, 1]
        predictions.append(fold)
    return pd.concat(predictions).sort_index()

def alert_metrics(y_true, score, threshold):
    alert = np.asarray(score) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, alert, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'true_positive': int(tp),
        'false_positive': int(fp),
        'true_negative': int(tn),
        'false_negative': int(fn),
        'hit_rate': tp / (tp + fn) if tp + fn else np.nan,
        'false_alarm_rate': fp / (fp + tn) if fp + tn else np.nan,
        'precision': precision_score(y_true, alert, zero_division=0),
    }

def select_baseline_threshold(oof):
    candidates = sorted(oof['hot35_days_to_date'].dropna().unique())
    metrics = pd.DataFrame([
        alert_metrics(oof['loss_flag'], oof['hot35_days_to_date'], threshold)
        for threshold in candidates
    ])
    metrics['youden_j'] = metrics['hit_rate'] - metrics['false_alarm_rate']
    return metrics.sort_values(
        ['youden_j', 'hit_rate', 'precision', 'threshold'],
        ascending=[False, False, False, True],
    ).iloc[0], metrics

def select_model_threshold(oof, baseline_far):
    metrics = pd.DataFrame([
        alert_metrics(oof['loss_flag'], oof['model_probability'], threshold)
        for threshold in MODEL_PROBABILITY_CANDIDATES
    ])
    eligible = metrics.loc[metrics['false_alarm_rate'].le(baseline_far + FAR_TOLERANCE)]
    if eligible.empty:
        eligible = metrics.assign(far_gap=(metrics['false_alarm_rate'] - baseline_far).abs())
        return eligible.sort_values(['far_gap', 'hit_rate'], ascending=[True, False]).iloc[0], metrics
    return eligible.sort_values(
        ['hit_rate', 'precision', 'false_alarm_rate', 'threshold'],
        ascending=[False, False, True, True],
    ).iloc[0], metrics


## 6. Tune on 2000–2018, then lock thresholds

For each crop, regularization strength is selected by out-of-fold Brier score. The baseline hot-day cutoff is selected from the same training-only predictions. The model alert threshold is then selected to maximize hit rate while keeping its false-alarm rate no worse than the baseline, within a small tolerance.

In [ ]:
tuning_rows = []
crop_settings = {}
train_oof = {}

for crop in CROPS:
    train = model_panel.loc[
        model_panel['crop'].eq(crop) & model_panel['year'].isin(TRAIN_YEARS)
    ].copy()

    for C in C_GRID:
        oof = out_of_fold_probabilities(train, C)
        tuning_rows.append({
            'crop': crop,
            'C': C,
            'oof_rows': len(oof),
            'brier_score': brier_score_loss(oof['loss_flag'], oof['model_probability']),
            'roc_auc': roc_auc_score(oof['loss_flag'], oof['model_probability']),
        })

    crop_tuning = pd.DataFrame(tuning_rows).query('crop == @crop').sort_values(['brier_score', 'roc_auc'], ascending=[True, False])
    selected_C = float(crop_tuning.iloc[0]['C'])
    oof = out_of_fold_probabilities(train, selected_C)
    baseline_choice, baseline_grid = select_baseline_threshold(oof)
    model_choice, model_grid = select_model_threshold(oof, baseline_choice['false_alarm_rate'])

    train_oof[crop] = oof
    crop_settings[crop] = {
        'C': selected_C,
        'baseline_threshold': float(baseline_choice['threshold']),
        'model_threshold': float(model_choice['threshold']),
        'baseline_oof_metrics': baseline_choice.to_dict(),
        'model_oof_metrics': model_choice.to_dict(),
        'baseline_grid': baseline_grid,
        'model_grid': model_grid,
    }

tuning_results = pd.DataFrame(tuning_rows).sort_values(['crop', 'brier_score'])
display(tuning_results)
display(pd.DataFrame({
    crop: {
        'selected_C': settings['C'],
        'baseline_hot35_threshold': settings['baseline_threshold'],
        'model_probability_threshold': settings['model_threshold'],
        'baseline_oof_hit_rate': settings['baseline_oof_metrics']['hit_rate'],
        'baseline_oof_false_alarm_rate': settings['baseline_oof_metrics']['false_alarm_rate'],
        'model_oof_hit_rate': settings['model_oof_metrics']['hit_rate'],
        'model_oof_false_alarm_rate': settings['model_oof_metrics']['false_alarm_rate'],
    }
    for crop, settings in crop_settings.items()
}).T)


## 7. Locked 2019–2025 evaluation

Nothing is retuned in this section. Each crop model is fitted once on 2000–2018, then compared with its locked 35°C-day rule on unseen 2019–2025 outcomes. A logistic model only earns deployment if it improves the operational comparison: higher hit rate at the same or lower false-alarm rate.

In [ ]:
evaluation_rows = []
test_predictions = []
evaluation_models = {}

for crop in CROPS:
    train = model_panel.loc[
        model_panel['crop'].eq(crop) & model_panel['year'].isin(TRAIN_YEARS)
    ].copy()
    test = model_panel.loc[
        model_panel['crop'].eq(crop) & model_panel['year'].isin(TEST_YEARS)
    ].copy()
    settings = crop_settings[crop]

    pipeline = make_pipeline(settings['C'])
    pipeline.fit(train[FEATURE_COLUMNS], train['loss_flag'])
    evaluation_models[crop] = pipeline

    test['model_probability'] = pipeline.predict_proba(test[FEATURE_COLUMNS])[:, 1]
    test['model_alert'] = test['model_probability'].ge(settings['model_threshold'])
    test['baseline_alert'] = test['hot35_days_to_date'].ge(settings['baseline_threshold'])
    test_predictions.append(test)

    baseline_metrics = alert_metrics(test['loss_flag'], test['hot35_days_to_date'], settings['baseline_threshold'])
    model_metrics = alert_metrics(test['loss_flag'], test['model_probability'], settings['model_threshold'])
    baseline_metrics.update({'crop': crop, 'method': '35°C-day baseline', 'roc_auc': roc_auc_score(test['loss_flag'], test['hot35_days_to_date']), 'brier_score': np.nan})
    model_metrics.update({'crop': crop, 'method': 'Logistic regression', 'roc_auc': roc_auc_score(test['loss_flag'], test['model_probability']), 'brier_score': brier_score_loss(test['loss_flag'], test['model_probability'])})
    evaluation_rows.extend([baseline_metrics, model_metrics])

test_predictions = pd.concat(test_predictions, ignore_index=True)
evaluation_results = pd.DataFrame(evaluation_rows)

# Overall evaluation uses each crop's already locked, crop-specific alert thresholds.
overall_baseline = alert_metrics(test_predictions['loss_flag'], test_predictions['hot35_days_to_date'], threshold=-np.inf)
overall_baseline.update({
    'crop': 'ALL',
    'method': '35°C-day baseline',
    'threshold': 'crop-specific',
    'roc_auc': roc_auc_score(test_predictions['loss_flag'], test_predictions['hot35_days_to_date']),
    'brier_score': np.nan,
})
overall_baseline.update(alert_metrics(test_predictions['loss_flag'], test_predictions['baseline_alert'].astype(int), threshold=1))
overall_model = alert_metrics(test_predictions['loss_flag'], test_predictions['model_alert'].astype(int), threshold=1)
overall_model.update({
    'crop': 'ALL',
    'method': 'Logistic regression',
    'threshold': 'crop-specific',
    'roc_auc': roc_auc_score(test_predictions['loss_flag'], test_predictions['model_probability']),
    'brier_score': brier_score_loss(test_predictions['loss_flag'], test_predictions['model_probability']),
})
evaluation_results = pd.concat([evaluation_results, pd.DataFrame([overall_baseline, overall_model])], ignore_index=True)

display(evaluation_results[['crop', 'method', 'threshold', 'hit_rate', 'false_alarm_rate', 'precision', 'roc_auc', 'brier_score']])


## 8. Interpret the result

The model beats the baseline only when the locked 2019–2025 results show a higher hit rate **and** the same or lower false-alarm rate. ROC-AUC reports ranking quality; Brier score reports probability quality. Coefficients are associations, not causal effects.

In [ ]:
comparison = evaluation_results.pivot(index='crop', columns='method', values=['hit_rate', 'false_alarm_rate', 'precision', 'roc_auc', 'brier_score'])
display(comparison)

for crop in CROPS:
    pipeline = evaluation_models[crop]
    coefficients = pd.DataFrame({
        'feature': FEATURE_COLUMNS,
        'standardized_coefficient': pipeline.named_steps['model'].coef_[0],
    })
    coefficients['odds_ratio_per_sd'] = np.exp(coefficients['standardized_coefficient'])
    print(f'\n{crop}: standardized model coefficients')
    display(coefficients.sort_values('standardized_coefficient', key=np.abs, ascending=False))

fig, axes = plt.subplots(1, len(CROPS), figsize=(12, 4), sharey=True)
for axis, crop in zip(np.ravel(axes), CROPS):
    subset = test_predictions.loc[test_predictions['crop'].eq(crop)]
    axis.hist(subset.loc[subset['loss_flag'].eq(0), 'model_probability'], bins=15, alpha=.65, label='No loss')
    axis.hist(subset.loc[subset['loss_flag'].eq(1), 'model_probability'], bins=15, alpha=.65, label='Loss')
    axis.axvline(crop_settings[crop]['model_threshold'], color='#b74638', linestyle='--', label='Alert threshold')
    axis.set(title=crop, xlabel='Predicted loss probability', ylabel='County-crop years')
    axis.legend(fontsize=8)
fig.suptitle('Unseen 2019–2025 model probabilities')
plt.tight_layout()


## 9. Refit approved configuration through 2025 and score 2026

Run this only after reviewing the locked test results. The configuration is frozen from the 2000–2018 selection process; the production fit may use all completed outcome years through 2025. The 2026 score is a current-season estimate, not a validated outcome.

In [ ]:
deployment_models = {}
current_features = all_features.loc[
    all_features['year'].isin(CURRENT_YEARS) & all_features['crop'].isin(CROPS)
].copy()
current_scored_parts = []

for crop in CROPS:
    completed_history = model_panel.loc[
        model_panel['crop'].eq(crop) & model_panel['year'].isin(TRAIN_YEARS + TEST_YEARS)
    ].copy()
    current = current_features.loc[current_features['crop'].eq(crop)].copy()
    pipeline = make_pipeline(crop_settings[crop]['C'])
    pipeline.fit(completed_history[FEATURE_COLUMNS], completed_history['loss_flag'])
    deployment_models[crop] = pipeline

    current['model_risk_probability'] = pipeline.predict_proba(current[FEATURE_COLUMNS])[:, 1]
    current['model_risk_score'] = (100 * current['model_risk_probability']).round(1)
    current['model_alert_threshold'] = crop_settings[crop]['model_threshold']
    current['model_alert'] = current['model_risk_probability'].ge(crop_settings[crop]['model_threshold'])
    current['baseline_hot35_threshold'] = crop_settings[crop]['baseline_threshold']
    current['baseline_hot35_alert'] = current['hot35_days_to_date'].ge(crop_settings[crop]['baseline_threshold'])
    current['data_mode'] = 'current'
    current['model_version'] = MODEL_VERSION
    current_scored_parts.append(current)

current_advisory = pd.concat(current_scored_parts, ignore_index=True)
current_advisory['model_risk_level'] = np.select(
    [current_advisory['model_alert'], current_advisory['model_risk_probability'].ge(0.20)],
    ['HIGH', 'WATCH'],
    default='LOW',
)

advisory_columns = [
    'fips', 'county_name', 'state_alpha', 'crop', 'year', 'as_of_date',
    'weather_through_date', 'drought_through_date', 'data_mode',
    'hot35_days_to_date', 'd2plus_weeks_to_date', 'max_drought_severity_to_date',
    'aws_100cm_mm', 'soil_storage_group',
    'hot35_x_drought', 'hot35_x_low_storage',
    'model_risk_probability', 'model_risk_score', 'model_risk_level',
    'model_alert_threshold', 'model_alert',
    'baseline_hot35_threshold', 'baseline_hot35_alert', 'model_version',
]
current_advisory = current_advisory[advisory_columns].sort_values(['crop', 'fips']).reset_index(drop=True)
display(current_advisory.head())
print(f'2026 scored rows: {len(current_advisory):,}')
print(current_advisory.groupby(['crop', 'model_risk_level']).size())


## 10. Optional exports

This cell is intentionally disabled by default. Set `WRITE_OUTPUTS = True` only after the model has passed review. It writes a 2026 advisory CSV, the locked evaluation table, and a JSON representation of the fitted model/scaler so a static React app can reproduce `predict_proba` for its Stress Simulator.

In [ ]:
if WRITE_OUTPUTS:
    output_dir = ROOT / 'data' / 'derived'
    output_dir.mkdir(parents=True, exist_ok=True)

    current_advisory.to_csv(output_dir / 'county_logistic_heat_advisory_2026.csv', index=False)
    evaluation_results.to_csv(output_dir / 'logistic_heat_model_evaluation_2019_2025.csv', index=False)

    artifacts = {
        'model_version': MODEL_VERSION,
        'target_definition': f'yield_anomaly_pct <= {LOSS_ANOMALY_THRESHOLD}',
        'feature_names': FEATURE_COLUMNS,
        'training_years_for_deployment': [min(TRAIN_YEARS + TEST_YEARS), max(TRAIN_YEARS + TEST_YEARS)],
        'crops': {},
    }
    for crop, pipeline in deployment_models.items():
        imputer = pipeline.named_steps['imputer']
        scaler = pipeline.named_steps['scaler']
        model = pipeline.named_steps['model']
        artifacts['crops'][crop] = {
            'C': crop_settings[crop]['C'],
            'alert_probability_threshold': crop_settings[crop]['model_threshold'],
            'baseline_hot35_threshold': crop_settings[crop]['baseline_threshold'],
            'imputer_statistics': imputer.statistics_.tolist(),
            'scaler_mean': scaler.mean_.tolist(),
            'scaler_scale': scaler.scale_.tolist(),
            'intercept': float(model.intercept_[0]),
            'coefficients': model.coef_[0].tolist(),
        }

    with open(output_dir / 'logistic_heat_model_parameters.json', 'w') as file:
        json.dump(artifacts, file, indent=2)

    print(f'Wrote model outputs to {output_dir}')
else:
    print('No files were written. Set WRITE_OUTPUTS = True only after review.')
